# Agri-Agent: Google Colab Edition
Run this notebook to set up and interact with the Agri-Agent using a free GPU.
**Note:** Ensure you have switched the Runtime to **GPU** (Runtime > Change runtime type > T4 GPU).

In [ ]:
# 1. Install System Dependencies
!apt-get install -y pciutils # Helper
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2. Start Ollama Server in Background
import subprocess
import time

# Start Ollama serve as a background process
print("Starting Ollama Server...")
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)  # Give it a few seconds to initialize
print("Ollama Server Started.")

In [ ]:
# 3. Pull Required Models
# The agent uses 'qwen3-vl' by default in agent_graph.py. 
# If that model is not available, you may need to change MODEL_NAME in agent_graph.py to 'qwen2.5-vl' or 'llama3.2-vision'
print("Pulling models... This may take a few minutes.")
!ollama pull qwen2.5-coder # Good for logic
!ollama pull llama3.2-vision # Good for vision
!ollama pull qwen3-vl # As requested in agent_graph.py
!ollama pull nomic-embed-text # For RAG embeddings


In [ ]:
# 4. Install Python Dependencies
!pip install -r requirements.txt
!pip install nest_asyncio pyngrok

In [ ]:
# 4a. START REMOTE TUNNEL (New!)
from pyngrok import ngrok
import os

# --- CONFIGURATION ---
# Sign up at ngrok.com and get your authtoken
# You can paste it below OR set it securely in the Secrets tab (key: NGROK_TOKEN)
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"   # <--- PASTE TOKEN HERE if not using Secrets

# Try to get from Colab Secrets
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
except:
    pass

if not NGROK_TOKEN or NGROK_TOKEN == "YOUR_NGROK_TOKEN_HERE":
    NGROK_TOKEN = input("Enter your ngrok Authtoken: ")

ngrok.set_auth_token(NGROK_TOKEN)

# Kill any existing tunnels to restart cleanly (Fixes ERR_NGROK_334)
ngrok.kill()

# Open a HTTP tunnel on port 11434 (Standard Ollama Port)
tunnel = ngrok.connect(11434)
public_url = tunnel.public_url
print(f"\n>>> REMOTE GPU URL: {public_url} <<<\n")
print("Copy the above URL and run this command locally:")
print(f"set OLLAMA_BASE_URL={public_url}")

In [ ]:
# 5. Run the Agent (Interactive Loop)
import sys
import os
import nest_asyncio

nest_asyncio.apply()

# Ensure we are in the right directory (adjust if you uploaded differently)
TARGET_DIR = '/content/agri-agent/agri-agent'
if not os.path.exists(TARGET_DIR):
    # Maybe we are already in the repo or root
    if os.path.exists("agent_graph.py"):
        TARGET_DIR = os.getcwd()
    else:
        print(f"Warning: Could not find {TARGET_DIR}. Please Make sure you uploaded the files.")
else:
    os.chdir(TARGET_DIR)

print(f"Working Directory: {os.getcwd()}")

try:
    from agent_graph import graph
    from langchain_core.messages import HumanMessage

    print("\nAgri-Agent Loaded Successfully! Type 'exit' to quit.")

    while True:
        user_input = input("\nUser (Ask me anything): ")
        if user_input.lower() in ['exit', 'quit']:
            break
        
        inputs = {"messages": [HumanMessage(content=user_input)]}
        
        print("Agent processing...")
        for event in graph.stream(inputs):
            for key, value in event.items():
                 if "messages" in value:
                     last_msg = value["messages"][-1]
                     if key == "Supervisor" and value.get("next") == "FINISH":
                         print(f"\n🤖 AGENT: {last_msg.content}\n")
                     elif key == "Librarian":
                         print(f"\n📚 Librarian: {last_msg.content}\n")
                     elif key == "Analyst":
                         print(f"\n📊 Analyst: {last_msg.content}\n")
except ImportError as e:
    print(f"Error importing agent: {e}. Did you install requirements?")
except Exception as e:
    print(f"Runtime Error: {e}")
